# Held-out validation — scoring

Compares the two coders' filled workbooks (`coder_A_blind.xlsx`, `coder_B_blind.xlsx`) and the
provider auditor's filled workbook (`provider_audit_blind.xlsx`) against `answer_key_PRIVATE.xlsx`,
produced by `validation_prep.ipynb`.

Run this **after** the coders and the auditor have filled in and returned their workbooks. Nothing in
this notebook needs the raw review corpus — it only needs the four filled/blind workbooks plus the
private key.

**Reported here (per the required validation design):**

- Per-construct (8 binary constructs): precision, recall, specificity, F1, Cohen's kappa, with
  bootstrap 95% CI — coder A vs. dictionary, coder B vs. dictionary, and coder A vs. coder B
  (inter-rater reliability).
- Sentiment (3-class: negative / neutral / positive): macro-F1, per-class precision/recall,
  balanced accuracy, Matthews correlation coefficient (MCC), confusion matrix, bootstrap 95% CI —
  each coder's independently-assigned sentiment vs. the rating-derived `sentiment_class` used in the
  analysis, plus coder A vs. coder B agreement.
- Provider query-stratum audit: fit rate with Wilson 95% interval, overall and per stratum — whether
  each provider is plausibly a hit for the query stratum that retrieved it. The stratum records which
  search returned the provider, not a verified organizational type, so this is face validity of the
  retrieval and not agreement against a ground truth.

All metrics are computed with hand-written functions (`metrics_lib.py`, generated by the first code
cell below) that reproduce scikit-learn's `f1_score`, `balanced_accuracy_score`, `cohen_kappa_score`
and `matthews_corrcoef` exactly (checked against scikit-learn during development) — this keeps the
notebook's dependencies identical to `requirements.txt` (no scikit-learn needed).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

# ---- paths: point these at your filled/blind workbooks and the private key -------------------
CODER_A_PATH = Path("coder_A_blind.xlsx")     # filled in by coder A
CODER_B_PATH = Path("coder_B_blind.xlsx")     # filled in by coder B
AUDIT_PATH   = Path("provider_audit_blind.xlsx")  # filled in by the provider auditor
KEY_PATH     = Path("answer_key_PRIVATE.xlsx")    # never share this file with coders/auditor

OUT_DIR = Path("scoring_output")
OUT_DIR.mkdir(exist_ok=True)

N_BOOT = 2000
SEED = 20260817
ALPHA = 0.05

CONSTRUCTS = ["disruption", "intermediation", "coordination", "delay",
              "digital", "price", "labour", "environmental"]
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
BINARY_LABELS = ["0", "1"]

print("Reading from:", Path.cwd())

In [ ]:
%%writefile metrics_lib.py
import numpy as np
import pandas as pd
import time


def confusion_counts(y_true, y_pred, labels):
    y_true = pd.Series(y_true).astype(str)
    y_pred = pd.Series(y_pred).astype(str)
    cm = pd.crosstab(y_true, y_pred, dropna=False).reindex(index=labels, columns=labels, fill_value=0)
    return cm


def prf_per_class(cm):
    labels = cm.index.tolist()
    out = {}
    total = cm.values.sum()
    for lab in labels:
        tp = cm.loc[lab, lab]
        fp = cm[lab].sum() - tp
        fn = cm.loc[lab].sum() - tp
        tn = total - tp - fp - fn
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else np.nan
        out[lab] = dict(precision=precision, recall=recall, specificity=specificity, f1=f1, support=int(tp + fn))
    return pd.DataFrame(out).T


def macro_f1(cm):
    return prf_per_class(cm)["f1"].mean()


def balanced_accuracy(cm):
    return prf_per_class(cm)["recall"].mean()


def cohen_kappa(cm):
    total = cm.values.sum()
    po = np.trace(cm.values) / total
    row_marg = cm.sum(axis=1).values / total
    col_marg = cm.sum(axis=0).values / total
    pe = (row_marg * col_marg).sum()
    if pe == 1:
        return np.nan
    return (po - pe) / (1 - pe)


def matthews_corrcoef_multiclass(cm):
    C = cm.values.astype(float)
    n = C.sum()
    c = np.trace(C)
    p_k = C.sum(axis=0)
    t_k = C.sum(axis=1)
    num = c * n - np.sum(p_k * t_k)
    den = np.sqrt((n**2 - np.sum(p_k**2)) * (n**2 - np.sum(t_k**2)))
    if den == 0:
        return np.nan
    return num / den


def _encode(y, labels):
    label_to_idx = {l: i for i, l in enumerate(labels)}
    return np.array([label_to_idx[str(v)] for v in y], dtype=np.int64)


def _batch_prf(cms):
    tp = np.diagonal(cms, axis1=1, axis2=2).astype(float)
    fp = cms.sum(axis=1) - tp
    fn = cms.sum(axis=2) - tp
    precision = np.where((tp + fp) > 0, tp / np.where((tp + fp) > 0, tp + fp, 1), np.nan)
    recall = np.where((tp + fn) > 0, tp / np.where((tp + fn) > 0, tp + fn, 1), np.nan)
    denom = precision + recall
    f1 = np.where(denom > 0, 2 * precision * recall / np.where(denom > 0, denom, 1), np.nan)
    return precision, recall, f1


def _batch_macro_f1(cms):
    _, _, f1 = _batch_prf(cms)
    return np.nanmean(f1, axis=1)


def _batch_balanced_accuracy(cms):
    _, recall, _ = _batch_prf(cms)
    return np.nanmean(recall, axis=1)


def _batch_cohen_kappa(cms):
    total = cms.sum(axis=(1, 2)).astype(float)
    po = np.diagonal(cms, axis1=1, axis2=2).sum(axis=1) / total
    row_marg = cms.sum(axis=2) / total[:, None]
    col_marg = cms.sum(axis=1) / total[:, None]
    pe = (row_marg * col_marg).sum(axis=1)
    return np.where(pe != 1, (po - pe) / np.where(pe != 1, 1 - pe, 1), np.nan)


def _batch_mcc(cms):
    C = cms.astype(float)
    n = C.sum(axis=(1, 2))
    c = np.diagonal(C, axis1=1, axis2=2).sum(axis=1)
    p_k = C.sum(axis=1)
    t_k = C.sum(axis=2)
    num = c * n - np.sum(p_k * t_k, axis=1)
    den = np.sqrt((n**2 - np.sum(p_k**2, axis=1)) * (n**2 - np.sum(t_k**2, axis=1)))
    return np.where(den != 0, num / np.where(den != 0, den, 1), np.nan)


def _batch_f1_pos(cms, pos_idx=1):
    _, _, f1 = _batch_prf(cms)
    return f1[:, pos_idx]


_BATCH_METRICS = {
    "kappa": _batch_cohen_kappa,
    "macro_f1": _batch_macro_f1,
    "balanced_accuracy": _batch_balanced_accuracy,
    "mcc": _batch_mcc,
    "f1_pos": _batch_f1_pos,
}


def bootstrap_ci(y_true, y_pred, metric, labels, n_boot=2000, seed=20260817, alpha=0.05):
    n_labels = len(labels)
    true_codes = _encode(y_true, labels)
    pred_codes = _encode(y_pred, labels)
    pair_codes = true_codes * n_labels + pred_codes
    n = len(pair_codes)
    n_sq = n_labels * n_labels

    rng = np.random.default_rng(seed)
    resample_idx = rng.integers(0, n, size=(n_boot, n))
    resampled_pairs = pair_codes[resample_idx]
    row_offset = (np.arange(n_boot) * n_sq)[:, None]
    flat = (resampled_pairs + row_offset).ravel()
    counts = np.bincount(flat, minlength=n_boot * n_sq)
    cms = counts.reshape(n_boot, n_labels, n_labels)

    batch_fn = _BATCH_METRICS[metric]
    stats = batch_fn(cms)
    stats = stats[~np.isnan(stats)]
    if len(stats) == 0:
        return np.nan, np.nan
    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return lo, hi


In [ ]:
from metrics_lib import (
    confusion_counts, prf_per_class, macro_f1, balanced_accuracy,
    cohen_kappa, matthews_corrcoef_multiclass, bootstrap_ci,
)

key_A = pd.read_excel(KEY_PATH, sheet_name="coder_A_key")
key_B = pd.read_excel(KEY_PATH, sheet_name="coder_B_key")
key_audit = pd.read_excel(KEY_PATH, sheet_name="provider_audit_key")

coder_A = pd.read_excel(CODER_A_PATH, sheet_name="Coding")
coder_B = pd.read_excel(CODER_B_PATH, sheet_name="Coding")
audit = pd.read_excel(AUDIT_PATH, sheet_name="Audit")

for name, filled, key, on in [
    ("coder A", coder_A, key_A, "validation_id"),
    ("coder B", coder_B, key_B, "validation_id"),
    ("provider audit", audit, key_audit, "provider_pseudo"),
]:
    missing = set(key[on]) - set(filled[on])
    print(f"{name}: {len(filled)} rows returned, {len(key)} expected, {len(missing)} missing")

# merge each filled workbook onto its private key
mA = key_A.merge(coder_A, on="validation_id", suffixes=("_key", "_coder"))
mB = key_B.merge(coder_B, on="validation_id", suffixes=("_key", "_coder"))
# NOTE: join coder A and coder B on review_id, not validation_id. Each coder's workbook assigns
# validation_id independently (rows are shown in a different random order per coder), so the same
# validation_id label does NOT refer to the same underlying review across coders A and B. review_id
# is the stable key that identifies "the same review coded twice".
mA_for_ab = mA[["review_id"] + [f"{c}_coder" for c in CONSTRUCTS] + ["sentiment"]].rename(
    columns={**{f"{c}_coder": c for c in CONSTRUCTS}}
)
mB_for_ab = mB[["review_id"] + [f"{c}_coder" for c in CONSTRUCTS] + ["sentiment"]].rename(
    columns={**{f"{c}_coder": c for c in CONSTRUCTS}}
)
mAB = mA_for_ab.merge(mB_for_ab, on="review_id", suffixes=("_A", "_B"))
m_audit = key_audit.merge(audit, on="provider_pseudo", suffixes=("_key", "_auditor"))

print("\nmerged coder A:", mA.shape, " merged coder B:", mB.shape, " audit:", m_audit.shape)

In [ ]:
# ---- per-construct binary metrics: coder vs. dictionary key, and coder A vs. coder B ---------

def score_binary(y_true, y_pred, label):
    cm = confusion_counts(y_true, y_pred, BINARY_LABELS)
    prf = prf_per_class(cm)
    kappa = cohen_kappa(cm)
    kappa_lo, kappa_hi = bootstrap_ci(y_true, y_pred, "kappa", BINARY_LABELS, N_BOOT, SEED, ALPHA)
    f1_lo, f1_hi = bootstrap_ci(y_true, y_pred, "f1_pos", BINARY_LABELS, N_BOOT, SEED, ALPHA)
    return dict(
        construct=label,
        precision_pos=prf.loc["1", "precision"], recall_pos=prf.loc["1", "recall"],
        specificity_pos=prf.loc["1", "specificity"], f1_pos=prf.loc["1", "f1"],
        f1_pos_ci_lo=f1_lo, f1_pos_ci_hi=f1_hi,
        kappa=kappa, kappa_ci_lo=kappa_lo, kappa_ci_hi=kappa_hi,
        support_pos=int(prf.loc["1", "support"]), n=len(y_true),
    )

rows_vs_dict_A, rows_vs_dict_B, rows_AB = [], [], []
for c in CONSTRUCTS:
    rows_vs_dict_A.append(score_binary(mA[f"{c}_key"].astype(int), mA[f"{c}_coder"].astype(int), c))
    rows_vs_dict_B.append(score_binary(mB[f"{c}_key"].astype(int), mB[f"{c}_coder"].astype(int), c))
    rows_AB.append(score_binary(mAB[f"{c}_A"].astype(int), mAB[f"{c}_B"].astype(int), c))

construct_vs_dict_A = pd.DataFrame(rows_vs_dict_A).round(3)
construct_vs_dict_B = pd.DataFrame(rows_vs_dict_B).round(3)
construct_interrater_AB = pd.DataFrame(rows_AB).round(3)

print("Coder A vs. dictionary\n", construct_vs_dict_A, "\n")
print("Coder B vs. dictionary\n", construct_vs_dict_B, "\n")
print("Coder A vs. coder B (inter-rater)\n", construct_interrater_AB)

construct_vs_dict_A.to_csv(OUT_DIR / "construct_metrics_coderA_vs_dictionary.csv", index=False)
construct_vs_dict_B.to_csv(OUT_DIR / "construct_metrics_coderB_vs_dictionary.csv", index=False)
construct_interrater_AB.to_csv(OUT_DIR / "construct_metrics_interrater_AB.csv", index=False)

In [ ]:
# ---- sentiment (3-class): coder's independent judgment vs. rating-derived sentiment_class ------

def score_sentiment(y_true, y_pred):
    cm = confusion_counts(y_true, y_pred, SENTIMENT_LABELS)
    prf = prf_per_class(cm)
    mf1 = macro_f1(cm)
    bacc = balanced_accuracy(cm)
    mcc = matthews_corrcoef_multiclass(cm)
    mf1_lo, mf1_hi = bootstrap_ci(y_true, y_pred, "macro_f1", SENTIMENT_LABELS, N_BOOT, SEED, ALPHA)
    bacc_lo, bacc_hi = bootstrap_ci(y_true, y_pred, "balanced_accuracy", SENTIMENT_LABELS, N_BOOT, SEED, ALPHA)
    mcc_lo, mcc_hi = bootstrap_ci(y_true, y_pred, "mcc", SENTIMENT_LABELS, N_BOOT, SEED, ALPHA)
    summary = pd.DataFrame([dict(
        macro_f1=mf1, macro_f1_ci_lo=mf1_lo, macro_f1_ci_hi=mf1_hi,
        balanced_accuracy=bacc, balanced_accuracy_ci_lo=bacc_lo, balanced_accuracy_ci_hi=bacc_hi,
        mcc=mcc, mcc_ci_lo=mcc_lo, mcc_ci_hi=mcc_hi, n=len(y_true),
    )]).round(3)
    return summary, prf.round(3), cm

mA["sentiment_norm"] = mA["sentiment"].astype(str).str.strip().str.lower()
mB["sentiment_norm"] = mB["sentiment"].astype(str).str.strip().str.lower()
mA["sentiment_class"] = mA["sentiment_class"].astype(str).str.strip().str.lower()
mB["sentiment_class"] = mB["sentiment_class"].astype(str).str.strip().str.lower()

sent_summary_A, sent_perclass_A, sent_cm_A = score_sentiment(mA["sentiment_class"], mA["sentiment_norm"])
sent_summary_B, sent_perclass_B, sent_cm_B = score_sentiment(mB["sentiment_class"], mB["sentiment_norm"])

mAB["sentiment_A_norm"] = mAB["sentiment_A"].astype(str).str.strip().str.lower()
mAB["sentiment_B_norm"] = mAB["sentiment_B"].astype(str).str.strip().str.lower()
sent_cm_AB = confusion_counts(mAB["sentiment_A_norm"], mAB["sentiment_B_norm"], SENTIMENT_LABELS)
sent_kappa_AB = cohen_kappa(sent_cm_AB)
sent_kappa_AB_lo, sent_kappa_AB_hi = bootstrap_ci(
    mAB["sentiment_A_norm"], mAB["sentiment_B_norm"], "kappa", SENTIMENT_LABELS, N_BOOT, SEED, ALPHA)

print("Coder A sentiment vs. rating-derived class\n", sent_summary_A, "\n", sent_perclass_A, "\n")
print("Confusion matrix (rows=rating-derived, cols=coder A):\n", sent_cm_A, "\n")
print("Coder B sentiment vs. rating-derived class\n", sent_summary_B, "\n", sent_perclass_B, "\n")
print("Confusion matrix (rows=rating-derived, cols=coder B):\n", sent_cm_B, "\n")
print(f"Coder A vs. coder B sentiment agreement: kappa={sent_kappa_AB:.3f} "
      f"[{sent_kappa_AB_lo:.3f}, {sent_kappa_AB_hi:.3f}]")

sent_summary_A.to_csv(OUT_DIR / "sentiment_metrics_coderA.csv", index=False)
sent_summary_B.to_csv(OUT_DIR / "sentiment_metrics_coderB.csv", index=False)
sent_perclass_A.to_csv(OUT_DIR / "sentiment_perclass_coderA.csv")
sent_perclass_B.to_csv(OUT_DIR / "sentiment_perclass_coderB.csv")
sent_cm_A.to_csv(OUT_DIR / "sentiment_confusion_coderA.csv")
sent_cm_B.to_csv(OUT_DIR / "sentiment_confusion_coderB.csv")
pd.DataFrame([dict(kappa=sent_kappa_AB, ci_lo=sent_kappa_AB_lo, ci_hi=sent_kappa_AB_hi,
                    n=len(mAB))]).round(3).to_csv(OUT_DIR / "sentiment_interrater_AB.csv", index=False)

In [ ]:
# ---- provider query-stratum audit: face validity of the retrieval ------------------------------
#
# This is NOT an agreement statistic against a ground-truth typology. The query stratum records which
# search returned the provider, so there is no independent "true segment" to agree with. What the
# assessor judges is fit: is this business plausibly a hit for the query stratum that retrieved it?
# Reported as a fit rate per stratum with a Wilson 95% interval.

FIT_LABELS = ["consistent", "inconsistent", "undetermined"]

m_audit["fit_norm"] = (
    m_audit["stratum_fit"].astype(str).str.strip().str.lower()
    .where(lambda s: s.isin(FIT_LABELS))
)
unmapped = m_audit["fit_norm"].isna().sum()
if unmapped:
    print(f"WARNING: {unmapped} audit rows have an unrecognized 'stratum_fit' value "
          f"(expected one of {FIT_LABELS}) and are excluded from scoring.")
m_audit_scored = m_audit.dropna(subset=["fit_norm"])


def wilson_interval(successes, total, z=1.96):
    """Wilson score interval for a proportion (no scipy dependency)."""
    if total == 0:
        return (np.nan, np.nan)
    p = successes / total
    denom = 1 + z**2 / total
    centre = (p + z**2 / (2 * total)) / denom
    halfwidth = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denom
    return (max(0.0, centre - halfwidth), min(1.0, centre + halfwidth))


def fit_row(frame, label):
    determined = frame[frame["fit_norm"] != "undetermined"]
    n_det = len(determined)
    n_consistent = int((determined["fit_norm"] == "consistent").sum())
    lo, hi = wilson_interval(n_consistent, n_det)
    return dict(
        stratum=label,
        n_audited=len(frame),
        n_undetermined=int((frame["fit_norm"] == "undetermined").sum()),
        n_determined=n_det,
        n_consistent=n_consistent,
        fit_rate=(n_consistent / n_det) if n_det else np.nan,
        fit_ci_lo=lo, fit_ci_hi=hi,
    )


rows = [fit_row(m_audit_scored, "all strata")]
for stratum, grp in m_audit_scored.groupby("query_stratum"):
    rows.append(fit_row(grp, stratum))
audit_summary = pd.DataFrame(rows).round(3)

audit_fit_table = pd.crosstab(m_audit_scored["query_stratum"], m_audit_scored["fit_norm"]).reindex(
    columns=FIT_LABELS, fill_value=0
)

print("Provider query-stratum audit \u2014 face validity of the retrieval\n", audit_summary, "\n")
print("Assessments by stratum (rows=query stratum, cols=assessor judgement):\n", audit_fit_table)
print("\nRead as: the share of providers a reasonable person would expect to find via the query "
      "stratum that retrieved them. It bounds how well the stratum stands in for the intended "
      "contrast; it says nothing about the providers' organizational form.")

audit_summary.to_csv(OUT_DIR / "provider_audit_fit.csv", index=False)
audit_fit_table.to_csv(OUT_DIR / "provider_audit_fit_by_stratum.csv")


In [ ]:
# ---- consolidated report ----------------------------------------------------------------------

report = {
    "n_construct_reviews": int(len(mA)),
    "n_providers_audited": int(len(m_audit_scored)),
    "construct_kappa_vs_dictionary_coderA": construct_vs_dict_A.set_index("construct")["kappa"].to_dict(),
    "construct_kappa_vs_dictionary_coderB": construct_vs_dict_B.set_index("construct")["kappa"].to_dict(),
    "construct_kappa_interrater_AB": construct_interrater_AB.set_index("construct")["kappa"].to_dict(),
    "sentiment_macro_f1_coderA": float(sent_summary_A["macro_f1"].iloc[0]),
    "sentiment_macro_f1_coderB": float(sent_summary_B["macro_f1"].iloc[0]),
    "sentiment_kappa_interrater_AB": float(sent_kappa_AB),
    "provider_audit_fit_rate_overall": float(audit_summary.loc[0, "fit_rate"]),
    "provider_audit_fit_ci": [float(audit_summary.loc[0, "fit_ci_lo"]), float(audit_summary.loc[0, "fit_ci_hi"])],
    "provider_audit_n_undetermined": int(audit_summary.loc[0, "n_undetermined"]),
}
with open(OUT_DIR / "validation_report_summary.json", "w") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(json.dumps(report, indent=2, ensure_ascii=False))
print(f"\nAll metric tables and this summary were written to {OUT_DIR.resolve()}")